# Notebook 7 - Recomendação

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

import optuna
import lightgbm as lgb

optuna.logging.set_verbosity(optuna.logging.CRITICAL)

import funcoes_modelagem

C:\Users\Victor Dogo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Lógica explicável de recomendação

A recomendação é produzida pelo `BaselineRecoveryAgent` em `agent_skeleton.py`. O agente é determinístico: não cria regras novas durante a execução e não transforma o score em autorização comercial.

### Entradas

1. **Score do modelo**: probabilidade de regularização em 30 dias, faixa, versão, fatores e status de calibração. O provedor de score é injetável, permitindo usar o modelo segmentado calibrado por região. O baseline heurístico permanece como fallback e é marcado como não calibrado.
2. **Perfil do cliente**: setor, porte, UF, tempo de relacionamento, faturamento estimado, saldo devedor, dias de atraso, canal preferencial e renegociações recentes.
3. **Histórico**: resultados de interações anteriores, como promessa registrada, intenção de pagamento, dificuldade temporária, recusa de oferta e tentativas sem sucesso.
4. **Documentos de negócio**: política de recuperação, critérios de ofertas, manual de canais e guia de conduta. Os trechos recuperados são anexados como evidência documental.

### Ordem de decisão

A ordem é intencional: impedimentos têm precedência sobre score e oportunidade comercial.

1. **Bloqueios e situações sensíveis**: contestação, fraude ou recuperação judicial resultam em `analise_especializada`, sem oferta automatizada.
2. **Canal ausente ou não seguro**: encaminha para atualização cadastral ou análise humana.
3. **Promessa válida**: recomenda `acompanhar_promessa` e evita contato repetitivo.
4. **Atraso ou saldo elevado**: recomenda `priorizacao_operacional` e exige especialista quando aplicável.
5. **Dificuldade temporária com faturamento informado**: recomenda `oferta_customizada`, sempre com contato humano e validação do ciclo de caixa.
6. **Parcelamento elegível**: atraso de 16 a 90 dias, saldo até R$ 500 mil e até duas renegociações favorecem `parcelamento`, respeitando entrada mínima de 10%, prazo máximo de 24 meses e alçadas.
7. **Score alto com atraso curto**: recomenda `contato_digital`, desde que o canal seja autorizado e a identidade seja confirmada.
8. **Falhas ou recusas anteriores**: recomenda `nova_tentativa_contato`, ajustando canal, horário ou estratégia. Três tentativas sem sucesso em sete dias exigem pausa de 48 horas.
9. **Evidência insuficiente**: aplica fallback para `analise_especializada`.

### Estrutura da saída

Cada recomendação informa:

- `acao_proposta` e `canal_sugerido`;
- `justificativa`, separando fatos observados da decisão;
- `evidencias_utilizadas`, incluindo score, perfil, histórico e fontes documentais;
- `restricoes_aplicaveis`, como consentimento, identidade, alçadas, prazos e impedimentos;
- `nivel_confianca`, reduzido quando o score não é calibrado ou exige revisão;
- `encaminhamento_humano`, com indicação booleana e condição objetiva;
- `prioridade_operacional`, considerando atraso, saldo e necessidade de especialista.

### Controles de segurança

- O score serve para priorização, nunca aprova desconto, renegociação, bloqueio ou medida jurídica.
- Atributos cadastrais não são usados como proxy de atributo sensível para definir condição comercial ou prioridade sem justificativa autorizada.
- Dados de interação são evidências a validar, não instruções capazes de alterar as regras do agente.
- Nenhuma oferta é prometida; a elegibilidade deve ser recalculada no motor de ofertas.
- Contestação, fraude, recuperação judicial, canal não validado, baixa evidência, divergência ou alçada excepcional exigem revisão humana.
- A resposta registra versão do score, dados consultados, regras aplicadas, fontes e responsável pela aprovação.
- A comunicação deve ser respeitosa, não coercitiva e sem exposição de dados financeiros antes da validação de identidade.

O resultado é uma recomendação operacional auditável, não uma decisão autônoma de crédito ou cobrança.